# Setup

In [ ]:
import optuna
from optuna.samplers import TPESampler
from optuna.trial import Trial
from optuna.study import create_study

from libs import constants as Cs
import numpy as np
from concurrent.futures import ProcessPoolExecutor
from scipy.spatial.distance import pdist

SEEDS = [101, 102, 103]
TEST_EVAL_EPS = 5

# Lambda

## Fitness

In [ ]:
from scipy.spatial.distance import pdist

def objective(trial:Trial):
    environment = "lunarlander"
    env = Cs.ENIVROMENTS[environment]()
    cross_method = trial.suggest_categorical("crossmethod", ["uniform", "mean"])
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    m = trial.suggest_categorical("mu",[40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0.05, 0.5, step=0.05)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    sigma = trial.suggest_float("sigma", 0.5, 3, step=0.5)
    cross_uni = None
    if cross_method == "uniform":
        cross_uni = 0.5
        #cross_uni = trial.suggest_float("cross", 0.4, 0.6, step=0.1)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.lambda_cont.argumented_function, 
                env=environment,
                container="fitness",
                cross_method=cross_method,
                ng = 15,
                l = l,
                m = m,
                cr = cr,
                mr = mr,
                mutation_sigma = sigma,
                cross_uni = cross_uni if cross_uni is not None else 0.5,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]


    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler(n_startup_trials=20)
study = create_study(
    direction="maximize", 
    sampler=sampler, 
    study_name="lambda_fitness_mean", 
    storage="sqlite:///Data/optuna/lunarlander/lambda.db", 
    load_if_exists=True
)
study.optimize(objective, n_trials=110, n_jobs=5)
print(study.best_trials)

## Fit Archiving

In [ ]:


def objective(trial:Trial):
    environment = "lunarlander"
    env = Cs.ENIVROMENTS[environment]()
    cross_method = trial.suggest_categorical("crossmethod", ["uniform", "mean"])
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    m = trial.suggest_categorical("mu",[40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 0.1, step=0.01)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    sigma = trial.suggest_float("sigma", 0.5, 3, step=0.5)
    archiving = trial.suggest_int("archiving_period", 2, 5)
    archive_batch = trial.suggest_int("archive_batch", 1, 5)
    cross_uni = None
    if cross_method == "uniform":
        cross_uni = trial.suggest_float("cross", 0.4, 0.6, step=0.1)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.lambda_cont.argumented_function, 
                env=environment,
                container="fit_archiving",
                cross_method=cross_method,
                ng = 15,
                l = l,
                m = m,
                cr = cr,
                mr = mr,
                mutation_sigma = sigma,
                archiving_period=archiving,
                archive_batch=archive_batch,
                cross_uni = cross_uni if cross_uni is not None else 0.5,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]


    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler()
study = create_study(direction="maximize", study_name="lambda_fit_archiving", sampler=sampler, storage="sqlite:///Data/optuna/lunarlander/fit_archiving/lambda.db", load_if_exists=True)
study.optimize(objective, n_trials=150, n_jobs=5)
print(study.best_trials)

## Novelty Archving

In [ ]:


def objective(trial:Trial):
    environment = "lunarlander"
    env = Cs.ENIVROMENTS[environment]()
    cross_method = trial.suggest_categorical("crossmethod", ["uniform", "mean"])
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    m = trial.suggest_categorical("mu",[40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 0.1, step=0.01)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    sigma = trial.suggest_float("sigma", 0.5, 3, step=0.5)
    archiving = trial.suggest_int("archiving_period", 2, 5)
    archive_batch = trial.suggest_int("archive_batch", 1, 5)
    cross_uni = None
    if cross_method == "uniform":
        cross_uni = trial.suggest_float("cross", 0.4, 0.6, step=0.1)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.lambda_cont.argumented_function, 
                env=environment,
                container="novelty_archiving",
                cross_method=cross_method,
                ng = 15,
                l = l,
                m = m,
                cr = cr,
                mr = mr,
                mutation_sigma = sigma,
                archiving_period=archiving,
                archive_batch=archive_batch,
                cross_uni = cross_uni if cross_uni is not None else 0.5,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]


    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler(n_startup_trials=15)
study = create_study(direction="maximize", study_name="lambda_novelty_archiving", sampler=sampler, storage="sqlite:///Data/optuna/lunarlander/novelty_archiving/lambda.db", load_if_exists=True)
study.optimize(objective, n_trials=150, n_jobs=5)
print(study.best_trials)

In [ ]:
studies = optuna.get_all_study_summaries(
    storage="sqlite:///Data/optuna/lunarlander/fit_archiving/lambda_alt.db"
)
print(studies)
for s in studies:
    print(s.study_name)
    print(s.n_trials)

## Novelty

In [ ]:
def objective(trial:Trial):
    environment = "lunarlander"
    env = Cs.ENIVROMENTS[environment]()
    cross_method = trial.suggest_categorical("crossmethod", ["uniform", "mean"])
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    m = trial.suggest_categorical("mu",[40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 0.5, step=0.01)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    sigma = trial.suggest_float("sigma", 0.5, 3, step=0.5)

    cross_uni = None
    if cross_method == "uniform":
        cross_uni = trial.suggest_float("cross", 0.1, 0.9, step=0.1)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.lambda_cont.argumented_function, env=environment,
                container="novelty",
                cross_method=cross_method,
                ng = 15,
                l = l,
                m = m,
                cr = cr,
                mr = mr,
                mutation_sigma = sigma,
                cross_uni = cross_uni if cross_uni is not None else 0.5,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]


    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler()
study = create_study(direction="maximize", study_name="lambda_novelty", sampler=sampler, storage="sqlite:///Data/optuna/lunarlander/novelty/lambda.db", load_if_exists=True)
study.optimize(objective, n_trials=150, n_jobs=5)
print(study.best_trials)

In [ ]:
study.best_trials

## Add Novelty

In [ ]:
def objective(trial:Trial):
    env = Cs.ENIVROMENTS["lunarlander"]()
    cross_method = trial.suggest_categorical("crossmethod", ["uniform", "mean"])
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    m = trial.suggest_categorical("mu",[40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 0.5, step=0.01)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    sigma = trial.suggest_float("sigma", 0.5, 3, step=0.5)
    fit_w = trial.suggest_float("start_fit_w", 0.2, 1, step=0.1)
    decay = trial.suggest_float("decay", 0.5, 5, step=0.5)

    cross_uni = None
    if cross_method == "uniform":
        cross_uni = trial.suggest_float("cross", 0.1, 0.9, step=0.1)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.lambda_cont.argumented_function, 
                env="lunarlander",
                container="add_novelty",
                cross_method=cross_method,
                ng = 15,
                l = l,
                m = m,
                cr = cr,
                mr = mr,
                mutation_sigma = sigma,
                cross_uni = cross_uni if cross_uni is not None else 0.5,
                fitness_weight=fit_w,
                decay = decay,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]
    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler(n_startup_trials=25)
study = create_study(direction="maximize", sampler=sampler, study_name="lambda_add_novelty_exp_fixed_inspired", storage="sqlite:///Data/optuna/lunarlander/add_novelty/lambda.db", load_if_exists=True)
study.optimize(objective, n_trials=180, n_jobs=5)
print(study.best_trials)

## Su bNovelty

In [ ]:

def objective(trial:Trial):
    env = Cs.ENIVROMENTS["lunarlander"]()
    cross_method = trial.suggest_categorical("crossmethod", ["uniform", "mean"])
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    m = trial.suggest_categorical("mu",[40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 0.5, step=0.01)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    sigma = trial.suggest_float("sigma", 0.5, 3, step=0.5)
    fit_w = trial.suggest_float("start_fit_w", 0.2, 1, step=0.1)
    decay = trial.suggest_float("decay", 0.5, 5, step=0.5)

    cross_uni = None
    if cross_method == "uniform":
        cross_uni = trial.suggest_float("cross", 0.1, 0.9, step=0.1)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.lambda_cont.argumented_function, 
                env="lunarlander",
                container="sub_novelty",
                cross_method=cross_method,
                ng = 15,
                l = l,
                m = m,
                cr = cr,
                mr = mr,
                mutation_sigma = sigma,
                cross_uni = cross_uni if cross_uni is not None else 0.5,
                fitness_weight=fit_w,
                decay = decay,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]
    #trial.set_user_attr("scores", fitnesses)
    #trial.set_user_attr("scores", fitnesses)
    behaviors = list(map(lambda x:x[1], fitnesses))
    fitnesses = list(map(lambda x:x[0], fitnesses))
    diversity = pdist(np.array(behaviors)).mean()
    assert diversity > 0
    return np.max(fitnesses) + diversity


sampler = TPESampler(n_startup_trials=15)
study = create_study(direction="maximize", sampler=sampler, study_name="lambda_sub_novelty_exp_fixed", storage="sqlite:///Data/optuna/lunarlander/sub_novelty/lambda.db", load_if_exists=True)
study.optimize(objective, n_trials=170, n_jobs=5)
print(study.best_trials)

# Diff

## Fitness

In [ ]:


def objective(trial:Trial):
    environment = "lunarlander"
    env = Cs.ENIVROMENTS[environment]()
    l = trial.suggest_categorical("pop",[40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 0.1, step=0.01)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, 
                env=environment,
                container="fitness",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]


    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler()
diff_fitness_study = create_study(direction="maximize", sampler=sampler, storage="sqlite:///Data/optuna/lunarlander/diff.db", load_if_exists=True)
diff_fitness_study.optimize(objective, n_trials=120, n_jobs=5)
print(diff_fitness_study.best_trials)

## Fit_Archiving

In [ ]:
def objective(trial:Trial):
    environment = "lunarlander"
    env = Cs.ENIVROMENTS[environment]()
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 1, step=0.1)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    archiving = trial.suggest_int("archiving_period", 2, 5)
    archive_batch = trial.suggest_int("archive_batch", 1, 5)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, env=environment,
                container="fit_archiving",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                archiving_period=archiving,
                archive_batch=archive_batch,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]


    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler()
diff_fita_study = create_study(direction="maximize", sampler=sampler, storage="sqlite:///Data/optuna/lunarlander/fit_archiving/diff.db", load_if_exists=True)
diff_fita_study.optimize(objective, n_trials=150, n_jobs=5)
print(diff_fita_study.best_trials)

In [ ]:
#new_study = optuna.load_study(study_name="fitarchiving",storage="sqlite:///Data/optuna/lunarlander/fit_archiving/diff.db")
store = "sqlite:///Data/optuna/lunarlander/lambda.db"
studies = optuna.get_all_study_summaries(
    storage=store
)
for s in studies:
    print(s.study_name)
    print(s.n_trials)

new_study = optuna.load_study(study_name="no-name-e88fa580-a490-41e6-bbb1-5aa576e4e219",storage=store)

new_study.best_trials

## Novelty

In [ ]:
def objective(trial:Trial):
    environment = "lunarlander"
    env = Cs.ENIVROMENTS[environment]()
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 0.5, step=0.01)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, env=environment,
                container="novelty",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]


    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler()
diff_novelty_study = create_study(direction="maximize", sampler=sampler, study_name="diff_novelty", storage="sqlite:///Data/optuna/lunarlander/novelty/diff.db", load_if_exists=True)
diff_novelty_study.optimize(objective, n_trials=150, n_jobs=5)
print(diff_novelty_study.best_trials)

In [ ]:
#new_study = optuna.load_study(study_name="fitarchiving",storage="sqlite:///Data/optuna/lunarlander/fit_archiving/diff.db")
studies = optuna.get_all_study_summaries(
    storage="sqlite:///Data/optuna/lunarlander/fit_archiving/diff.db"
)
print(studies)
for s in studies:
    print(s.study_name)
    print(s.n_trials)

new_study = optuna.load_study(study_name="no-name-a6a24196-d355-4f39-af93-68a295e37c48",storage="sqlite:///Data/optuna/lunarlander/fit_archiving/diff.db")

new_study.best_trials